In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import VGG19_Weights
import matplotlib.pyplot as plt
from torchvision.utils import save_image

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive/')
directory = '/content/drive/MyDrive/style_transfer/'

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
def load_images(loader, img_name):
    img = Image.open(img_name)
    img = loader(img).unsqueeze(0)
    return img.to(device)

In [ ]:
size = 256
crop = 224

img_loader = transforms.Compose([
    transforms.Resize(size),
    transforms.CenterCrop(crop),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

folders = ['content_pictures', 'style_pictures', 'generated']
content_imgs = []
style_imgs = []
gen_imgs = []

content_pictures = os.path.join(directory, folders[0])
style_pictures = os.path.join(directory, folders[1])
generated = os.path.join(directory, folders[2])


for item in os.listdir(content_pictures):
  img_path = os.path.join(content_pictures, item)
  content_imgs.append(load_images(img_loader, img_path))
  gen_imgs.append(load_images(img_loader, os.path.join(generated, item)).requires_grad_(True))

for index, item in enumerate(os.listdir(style_pictures)):
  img_path = os.path.join(style_pictures, item)
  style_imgs.append(load_images(img_loader, img_path))
# content_img = load_images(img_loader, os.path.join(directory, "golden_la.jpeg"))
# style_img = load_images(img_loader, os.path.join(directory, "stary_night.jpeg"))
# gen_img = load_images(img_loader, os.path.join(directory, "gen_img.png")).requires_grad_(True)


In [ ]:
learning_rate = 0.03
li = [0, 5, 10, 19, 28]

# optimizers = []
# for gen_img in gen_imgs:
#   optimizers.append(optim.Adam([gen_img], lr=learning_rate))


min_loss = float('inf')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
class VGG19(nn.Module):
    def __init__(self):
        super(VGG19, self).__init__()
        self.model = models.vgg19(pretrained=True).features.to(device).eval()

    def forward(self, x):
        features = []
        for index, layer in enumerate(self.model):
            x = layer(x)
            if index in li:
                features.append(x)
        return features

In [ ]:
model = VGG19()
n_steps = 2000

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
alpha = 1
beta = 0.01
best_imgs = []

for i in range(10):
  optimizer = optim.Adam([gen_imgs[i]], lr=learning_rate)
  best_loss = float('inf')
  best_img = None
  for epoch in range(n_steps):
      content_features = model(content_imgs[i])
      style_features = model(style_imgs[i])
      gen_features = model(gen_imgs[i])

      optimizer.zero_grad()
      content_loss = 0
      style_loss = 0

      for layer_content, layer_style, layer_gen in zip(content_features, style_features, gen_features):
          # Content Loss Calculation
          C = layer_content.reshape(1, -1)
          A = layer_gen.reshape(1, -1)
          content_loss += torch.mean((C-A)**2, 1)

          # Style Loss Calculation
          S = layer_style.reshape(layer_style.shape[1], -1)
          G = layer_gen.reshape(layer_gen.shape[1], -1)
          gram_S = torch.matmul(S, S.t())
          gram_gen = torch.matmul(G, G.t())
          style_loss += (torch.mean((gram_S-gram_gen)**2))

      # Combine content and style losses
      loss = alpha * content_loss + beta * style_loss

      # Calculate combined loss for the generated image
      combined_loss = loss.item()

      # Update best loss and corresponding image
      if combined_loss < best_loss:
          best_loss = combined_loss
          best_img = gen_imgs[i].clone()

      print(f"Loss at epoch {epoch+1} is ", combined_loss)
      loss.backward()
      optimizer.step()

      if epoch % 200 == 0:
          save_image(gen_imgs[i], os.path.join(directory, "generated", f"gen_img_{i + 1}_{epoch+1}.png"))
  best_imgs.append(best_img)


Выходные данные были обрезаны до нескольких последних строк (5000).
Loss at epoch 1001 is  1482.4287109375
Loss at epoch 1002 is  1482.203857421875
Loss at epoch 1003 is  1481.5048828125
Loss at epoch 1004 is  1478.726318359375
Loss at epoch 1005 is  1474.3189697265625
Loss at epoch 1006 is  1469.7274169921875
Loss at epoch 1007 is  1467.229248046875
Loss at epoch 1008 is  1466.640380859375
Loss at epoch 1009 is  1466.18505859375
Loss at epoch 1010 is  1464.723388671875
Loss at epoch 1011 is  1461.7396240234375
Loss at epoch 1012 is  1458.2657470703125
Loss at epoch 1013 is  1455.365966796875
Loss at epoch 1014 is  1453.6829833984375
Loss at epoch 1015 is  1452.7572021484375
Loss at epoch 1016 is  1451.5633544921875
Loss at epoch 1017 is  1449.736572265625
Loss at epoch 1018 is  1446.904296875
Loss at epoch 1019 is  1443.899658203125
Loss at epoch 1020 is  1441.6312255859375
Loss at epoch 1021 is  1440.68017578125
Loss at epoch 1022 is  1440.4632568359375
Loss at epoch 1023 is  1439.91

In [ ]:
print("Image with the least loss:")

for i, img in enumerate(best_imgs):
  save_image(img, os.path.join(directory, f"best_image_{i}.png"))

Image with the least loss:
